# Frozen AudioSet AST replacement, fixed experiment 32

Pinned AST checkpoint, native 16 kHz audio, fixed two-second clips and 0.5-second
centers; float16 frozen inference, float32 cache/training. No audio fine-tuning or
new annotations. PCA-32 and scaling fit separately on remaining known training
frames within five whole-window folds, with 60-second purge and three seeds.
Only the OOF-selected policy is scored on reused main; all assets remain private.


In [ ]:
from pathlib import Path
import hashlib,json,os,subprocess,sys,torch
assert torch.cuda.is_available()
print('GPU:',torch.cuda.get_device_name(0),flush=True)
INPUT=Path('/kaggle/input');WORK=Path('/kaggle/working/possession_ast');WORK.mkdir(parents=True,exist_ok=True)
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
configs=list(INPUT.rglob('ast_input.json'));assert len(configs)==1
FILES=configs[0].parent;plan=json.loads(configs[0].read_text())
roots=[p.parent for p in INPUT.rglob('F1_TARGET.md') if (p.parent/'temporal_possession/ast_study.py').is_file()];assert len(roots)==1
CODE=roots[0]
for relative,digest in plan['source_sha256'].items():assert sha(CODE/relative)==digest
baseplans=list(INPUT.rglob('clap_input.json'));assert len(baseplans)==1;BASE=baseplans[0].parent
FEATURES=BASE/'features.npz';GRAPH=BASE/'graph.npz';OWNER=BASE/'owner_scores.npz';REFERENCE=BASE/'reference_audio.npz'
for name,path in [('features',FEATURES),('graph',GRAPH),('owner_scores',OWNER),('reference_audio',REFERENCE)]:assert sha(path)==plan['fingerprints'][name]
sources=[p.parent for p in INPUT.rglob('experiment.json') if json.loads(p.read_text()).get('kind')=='team_symmetric_graph_memory_span_ablation'];assert len(sources)==1
SOURCE=sources[0]
controls=[p.parent for p in INPUT.rglob('decision.json') if p.parent.name=='ast_control'];assert len(controls)==1;CONTROL=controls[0]
for relative,digest in plan['control_sha256'].items():assert sha(CONTROL/relative)==digest
labels=[p.parent for p in INPUT.rglob('train.csv') if p.parent.name=='clap_labels'];assert len(labels)==1;LABELS=labels[0]
for name in ('train','validation'):assert sha(LABELS/f'{name}.csv')==plan['fingerprints'][name+'_labels']
videos=list(INPUT.rglob('BAR-ATH_tactical.mp4'));assert len(videos)==1;VIDEO=videos[0];assert sha(VIDEO)==plan['video_sha256']
DEPS=Path('/tmp/barca_ast_python')
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--target',str(DEPS),'-r',str(CODE/'requirements-temporal.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--target',str(DEPS),'--no-deps','-r',str(CODE/'requirements-foundation.txt')],check=True)
env=dict(os.environ,PYTHONPATH=str(DEPS)+os.pathsep+str(CODE),PYTHONDONTWRITEBYTECODE='1',HF_HUB_OFFLINE='1',HF_HUB_DISABLE_XET='1',CUBLAS_WORKSPACE_CONFIG=':4096:8')
def run(*args,network=False):
 current=dict(env)
 if network:current['HF_HUB_OFFLINE']='0'
 subprocess.run([sys.executable,*map(str,args)],cwd=CODE,env=current,check=True)
run('-m','unittest','discover','-s','tests','-v')
print('Full suite passed; starting fixed AST extraction.',flush=True)


In [ ]:
AUDIO=WORK/'ast_audio.npz'
run('-m','temporal_possession.ast_audio','--video',VIDEO,'--reference-audio',REFERENCE,'--out',AUDIO,'--device','cuda','--batch-size','16',network=True)


In [ ]:
RUN=WORK/'study'
run('-m','temporal_possession.ast_study','--features',FEATURES,'--audio',AUDIO,'--graph',GRAPH,'--owner-scores',OWNER,
 '--source-study',SOURCE,'--control-study',CONTROL,'--train-labels',LABELS/'train.csv','--validation-labels',LABELS/'validation.csv',
 '--out',RUN,'--device','cuda','--bootstrap','1000')


In [ ]:
report=json.loads((RUN/'report.json').read_text());assert not report['experiment']['validation_used_for_selection']
protocol=dict(git_checkpoint=plan['git_checkpoint'],source_sha256=plan['source_sha256'],fingerprints=plan['fingerprints'],
 control_sha256=plan['control_sha256'],torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),
 audio_sha256=sha(AUDIO),selected_csv_sha256=sha(RUN/'selected_predictions.csv'),audio_finetuned=False,
 reference_files_unchanged=all(sha(LABELS/f'{n}.csv')==plan['fingerprints'][n+'_labels'] for n in ('train','validation')))
assert protocol['reference_files_unchanged']
(WORK/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')
print('AST GPU RUN COMPLETE',flush=True)
